In [7]:
import pandas as pd
import sys
from pathlib import Path

PROJECT_DIR = Path("/mnt/common-ssd/karakasvioletta/valasztokeruletek")

sys.path.insert(0, str(PROJECT_DIR))

from config import PROCESSED_DATA_DIR

Különbségek vizsgálata település szinten (a pozitív különbség azt jelenti, hogy a 2026-os adatbázisban van több párosítatlan cím az adott településen)

In [13]:
telepulesek = pd.read_csv(
    PROCESSED_DATA_DIR / "differences" / "01_telepulesenkent.csv"
)

pir = pd.read_csv(
    PROCESSED_DATA_DIR / "differences" / "02_pir_szerint.csv"
)

kozterulet = pd.read_csv(
    PROCESSED_DATA_DIR / "differences" / "03_kozterulet_neve_szerint.csv"
)

In [12]:
pozitiv = (telepulesek["kulonbseg"] > 0).sum()
negativ = (telepulesek["kulonbseg"] < 0).sum()
nulla = (telepulesek["kulonbseg"] == 0).sum()

print(f"Pozitív különbségek száma: {pozitiv:,}")
print(f"Negatív különbségek száma: {negativ:,}")
print(f"Nulla különbségek száma: {nulla:,}")

Pozitív különbségek száma: 1,800
Negatív különbségek száma: 941
Nulla különbségek száma: 113


In [16]:
nullas_telepulesek = telepulesek[
    telepulesek["kulonbseg"] == 0
].copy()

nullas_telepulesek_lista = nullas_telepulesek["település_azonosító"].tolist()

Azokon a településeken, ahol ugyanannyi 22-es és 26-os párosítatlan cím volt, tovább mentem az irányítószám szintre és ugyanúgy megnéztem a különbségeket

In [17]:
nullas_telepulesek_pir = pir[
    pir["település_azonosító"].isin(
        nullas_telepulesek["település_azonosító"]
    )
].copy()

pozitiv = (nullas_telepulesek_pir["kulonbseg"] > 0).sum()
negativ = (nullas_telepulesek_pir["kulonbseg"] < 0).sum()
nulla = (nullas_telepulesek_pir["kulonbseg"] == 0).sum()

print(f"Pozitív PIR-különbségek: {pozitiv:,}")
print(f"Negatív PIR-különbségek: {negativ:,}")
print(f"Nulla PIR-különbségek: {nulla:,}")
print(f"Összes PIR-sor: {len(nullas_telepulesek_pir):,}")

Pozitív PIR-különbségek: 6
Negatív PIR-különbségek: 5
Nulla PIR-különbségek: 108
Összes PIR-sor: 119


In [20]:
nullas_pir = nullas_telepulesek_pir[
    nullas_telepulesek_pir["kulonbseg"] == 0
].copy()

In [23]:
nullas_pir_kozterulet = kozterulet.merge(
    nullas_pir[["település_azonosító", "PIR"]],
    on=["település_azonosító", "PIR"],
    how="inner"
)


# 3. Pozitív, negatív és nulla különbségek megszámolása
pozitiv = (nullas_pir_kozterulet["kulonbseg"] > 0).sum()
negativ = (nullas_pir_kozterulet["kulonbseg"] < 0).sum()
nulla = (nullas_pir_kozterulet["kulonbseg"] == 0).sum()

print(f"Pozitív közterület-különbségek: {pozitiv:,}")
print(f"Negatív közterület-különbségek: {negativ:,}")
print(f"Nulla közterület-különbségek: {nulla:,}")
print(f"Összes közterületi sor: {len(nullas_pir_kozterulet):,}")

Pozitív közterület-különbségek: 182
Negatív közterület-különbségek: 187
Nulla közterület-különbségek: 73
Összes közterületi sor: 442


In [24]:
egyezesek = pd.read_csv(
    PROCESSED_DATA_DIR / "differences" / "egyezesek.csv"
)

In [25]:
osszeg_2022 = egyezesek["csak_2022_db"].sum()

print(f"A csak_2022_db oszlop összege: {osszeg_2022:,}")

A csak_2022_db oszlop összege: 102


Azok közül, amiket össze lehetett párosítani hány cím szerepel a normalizált egyezések között

In [1]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

# Fájlok elérési útja
PROJECT_DIR = Path("/mnt/common-ssd/karakasvioletta/valasztokeruletek")

uj_parositas_path = (
    PROJECT_DIR / "data" / "processed" / "differences"
    / "egyezesek_normalizalt" / "egyezo_cimek.csv"
)

regi_egyezesek_path = (
    PROJECT_DIR / "data" / "processed" / "comparison"
    / "normalizalt_egyezesek.csv"
)

# Fájlok beolvasása
uj_parositas = pd.read_csv(
    uj_parositas_path,
    dtype=str,
    keep_default_na=False
)

regi_egyezesek = pd.read_csv(
    regi_egyezesek_path,
    dtype=str,
    keep_default_na=False
)


# Közterületnevek normalizálása
def szoveg_normalizalasa(ertek):
    ertek = str(ertek).lower().strip()

    # Ékezetek eltávolítása
    ertek = unicodedata.normalize("NFD", ertek)
    ertek = "".join(
        karakter for karakter in ertek
        if unicodedata.category(karakter) != "Mn"
    )

    # Írásjelek eltávolítása, szóközök egységesítése
    ertek = re.sub(r"[^a-z0-9]", " ", ertek)
    ertek = re.sub(r"\s+", " ", ertek).strip()

    return ertek


# Összehasonlításhoz szükséges kulcsok létrehozása
uj_parositas["_kozterulet_norm"] = (
    uj_parositas["közterület_neve"]
    .apply(szoveg_normalizalasa)
)

uj_parositas["_hazszam_norm"] = (
    uj_parositas["házszám_normalizált"].str.strip()
)

regi_egyezesek["_kozterulet_norm"] = (
    regi_egyezesek["közterület_neve_norm"].apply(szoveg_normalizalasa)
)

regi_egyezesek["_hazszam_norm"] = (
    regi_egyezesek["házszám_norm"].str.strip()
)


# Összehasonlítási kulcs
KULCSOK = [
    "település_azonosító",
    "_kozterulet_norm",
    "_hazszam_norm"
]

# A régi táblában csak az egyedi kulcsokat tartjuk meg,
# hogy az összevetés ne sokszorozza meg az új párosítás sorait.
regi_kulcsok = regi_egyezesek[KULCSOK].drop_duplicates()

# Megnézzük, mely új párosítások szerepelnek a régi táblában
eredmeny = uj_parositas.merge(
    regi_kulcsok.assign(szerepel_a_regi_tablaban=True),
    on=KULCSOK,
    how="left"
)

eredmeny["szerepel_a_regi_tablaban"] = (
    eredmeny["szerepel_a_regi_tablaban"].fillna(False)
)

# Összesítés
osszes = len(eredmeny)
egyezo = eredmeny["szerepel_a_regi_tablaban"].sum()
nem_egyezo = osszes - egyezo

print(f"Új párosítások száma: {osszes:,}")
print(f"A régi normalizált egyezések között is szerepel: {egyezo:,}")
print(f"Nem szerepel a régi táblában: {nem_egyezo:,}")

if osszes > 0:
    print(f"Arány: {egyezo / osszes * 100:.2f}%")

Új párosítások száma: 39
A régi normalizált egyezések között is szerepel: 4
Nem szerepel a régi táblában: 35
Arány: 10.26%


/tmp/ipykernel_4076128/1188775781.py:89: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  eredmeny["szerepel_a_regi_tablaban"].fillna(False)
